In [2]:
from os import path
import warnings
import polars as pl
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import numpy as np

warnings.filterwarnings("ignore")

tabular_data_path = path.join("..", "..", "data", "tabular")
processed_csv = path.join(tabular_data_path, "processed.csv")
target_col = "GUY SCORE"

df = pl.read_csv(processed_csv)

print("=" * 60)
print("GSS PREDICTION - ONE-VS-REST XGBOOST")
print("=" * 60)

gss_dist = df[target_col].value_counts().sort("GUY SCORE")
print("\nGUY SCORE Distribution:")
print(gss_dist)

preop_features = [
    "YAŞ",
    "CİNSİYET",
    "TARAF",
    "LOKALİZASYON",
    "TOPLAM TAŞ YÜKÜ (CM2)",
    "SOLİTER BB",
    "RENAL ANOMALİ",
    "EK RENAL HASTALIK",
    "GEÇİRİLMİŞ CERRAHİ",
    "TAS ANAMNEZI",
    "İKAB",
    "KREATİNİN",
    "ASA SKORU",
    "BT",
    "ÖZGEÇMİŞ",
]

preop_features = [c for c in preop_features if c in df.columns]

X = df.select(preop_features).to_numpy()
y = df[target_col].cast(pl.Int64, strict=False).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=492, stratify=y
)

imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "scale_pos_weight": [1, 2, 3],
}

grade_classifiers = {}
grade_metrics = {}

for grade in [1, 2, 3, 4]:
    print(f"\n{'=' * 60}")
    print(f"Training Classifier for Grade {grade} vs Rest")
    print("=" * 60)

    y_train_binary = (y_train == grade).astype(int)
    y_test_binary = (y_test == grade).astype(int)

    xgb_model = xgb.XGBClassifier(
        random_state=492,
        eval_metric="logloss",
        tree_method="hist",
    )

    grid_search = GridSearchCV(
        xgb_model,
        param_grid,
        cv=5,
        scoring="f1",
        n_jobs=-1,
        verbose=0,
    )

    grid_search.fit(X_train_scaled, y_train_binary)

    best_model = grid_search.best_estimator_
    grade_classifiers[grade] = best_model

    y_pred_binary = best_model.predict(X_test_scaled)

    cv_scores = cross_val_score(best_model, X_train_scaled, y_train_binary, cv=5, scoring="f1")

    grade_metrics[grade] = {
        "accuracy": accuracy_score(y_test_binary, y_pred_binary),
        "precision": precision_score(y_test_binary, y_pred_binary, zero_division=0),
        "recall": recall_score(y_test_binary, y_pred_binary, zero_division=0),
        "f1": f1_score(y_test_binary, y_pred_binary, zero_division=0),
        "cv_f1_mean": cv_scores.mean(),
        "cv_f1_std": cv_scores.std(),
        "best_params": grid_search.best_params_,
        "support": y_test_binary.sum(),
    }

    print(f"Best Params: {grid_search.best_params_}")
    print(f"Accuracy: {grade_metrics[grade]['accuracy']:.3f}")
    print(f"Precision: {grade_metrics[grade]['precision']:.3f}")
    print(f"Recall: {grade_metrics[grade]['recall']:.3f}")
    print(f"F1 Score: {grade_metrics[grade]['f1']:.3f}")
    print(f"5-Fold CV F1: {grade_metrics[grade]['cv_f1_mean']:.3f} (+/- {grade_metrics[grade]['cv_f1_std']:.3f})")

print("\n" + "=" * 60)
print("OVR RESULTS SUMMARY")
print("=" * 60)

summary_rows = []
for grade in [1, 2, 3, 4]:
    m = grade_metrics[grade]
    summary_rows.append({
        "Grade": grade,
        "Accuracy": m["accuracy"],
        "Precision": m["precision"],
        "Recall": m["recall"],
        "F1": m["f1"],
        "CV F1": m["cv_f1_mean"],
        "Support": m["support"],
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

print("\n" + "=" * 60)
print("COMPARISON")
print("=" * 60)

avg_f1 = np.mean([grade_metrics[g]["f1"] for g in [1, 2, 3, 4]])
avg_cv_f1 = np.mean([grade_metrics[g]["cv_f1_mean"] for g in [1, 2, 3, 4]])

print(f"Average F1 (OvR): {avg_f1:.3f}")
print(f"Average CV F1 (OvR): {avg_cv_f1:.3f}")

print("""
Previous Multiclass XGBoost:
  Accuracy: 0.8148
  Weighted F1: 0.8072
  CV F1: 0.8259 (+/- 0.0487)
""")

if avg_f1 > 0.80:
    print("✓ OvR shows competitive performance")
else:
    print("⚠ OvR does not significantly improve over multiclass")

GSS PREDICTION - ONE-VS-REST XGBOOST

GUY SCORE Distribution:
shape: (4, 2)
┌───────────┬───────┐
│ GUY SCORE ┆ count │
│ ---       ┆ ---   │
│ i64       ┆ u32   │
╞═══════════╪═══════╡
│ 1         ┆ 65    │
│ 2         ┆ 116   │
│ 3         ┆ 41    │
│ 4         ┆ 47    │
└───────────┴───────┘

Training Classifier for Grade 1 vs Rest
Best Params: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 100, 'scale_pos_weight': 1, 'subsample': 1.0}
Accuracy: 0.907
Precision: 0.900
Recall: 0.692
F1 Score: 0.783
5-Fold CV F1: 0.885 (+/- 0.082)

Training Classifier for Grade 2 vs Rest
Best Params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 100, 'scale_pos_weight': 1, 'subsample': 1.0}
Accuracy: 0.852
Precision: 0.778
Recall: 0.913
F1 Score: 0.840
5-Fold CV F1: 0.833 (+/- 0.050)

Training Classifier for Grade 3 vs Rest
Best Params: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'scale_pos_we